# Long/short portfolio ($\lVert w\rVert_1 = 1$)

Same 3-month window as before, but now we allow **long and short** with finite capital: the constraint is **gross exposure** $\sum_i |w_i| = 1$ (one dollar of gross, split between longs and shorts). We drop the softmax and instead set $w = z / \lVert z\rVert_1$ for a free signed vector $z$, which enforces $\lVert w\rVert_1 = 1$ while letting weights be negative. We solve **both** objectives: the utility $\mu^\top w - \tfrac{\gamma}{2}w^\top\Sigma w$ and the ratio $\mu^\top w / w^\top\Sigma w$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from stonks import get_prices, to_returns

%matplotlib inline


## Parameters


In [ ]:
TOP_N = 500
MONTHS = 3
PERIOD = "2y"      # cached; slice the last MONTHS
INTERVAL = "1d"
ALPHA = 1.0        # jitter lambda = ALPHA * mean(diag(Sigma))
gamma = 2.0        # risk aversion (utility objective)
torch.manual_seed(0)


## Fetch, slice, compute $\mu$ and $\Sigma$ (jittered)


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval=INTERVAL, field="close")
cut = pd.Timestamp(prices.columns[-1]) - pd.DateOffset(months=MONTHS)
returns = to_returns(prices.loc[:, prices.columns >= cut]).dropna()

X = returns.to_numpy(dtype=float)
tickers = list(returns.index)
N, T = X.shape
mu = X.mean(axis=1)
Xc = X - mu[:, None]
Sigma = (Xc @ Xc.T) / (T - 1)

lam = ALPHA * np.mean(np.diag(Sigma))
Sigma_jit = Sigma + lam * np.eye(N)          # jitter -> PD (ridge)
mu_t = torch.tensor(mu, dtype=torch.float64)
Sjit_t = torch.tensor(Sigma_jit, dtype=torch.float64)

print(f"N={N}, T={T}, rank(Sigma)={np.linalg.matrix_rank(Sigma)}, cond(Sigma_jit)={np.linalg.cond(Sigma_jit):.1e}")


## Enforcing $\lVert w\rVert_1 = 1$

$w = z / \lVert z\rVert_1 = z / \sum_i |z_i|$. This fixes gross exposure at 1 and lets each $w_i$ be signed (long if $z_i>0$, short if $z_i<0$). **Net exposure** $\sum_i w_i \in [-1,1]$ is free — near 0 means market-neutral.


In [ ]:
def l1norm(z):
    return z / z.abs().sum()

def report(name, w):
    sharpe = (w @ mu) * 252 / np.sqrt(w @ Sigma @ w * 252)   # measured on the real Sigma
    print(f"{name}: ||w||_1={np.abs(w).sum():.4f}  net={w.sum():+.3f}  "
          f"longs={int((w > 1e-4).sum())}  shorts={int((w < -1e-4).sum())}  Sharpe={sharpe:.2f}")


## Version A — utility objective

Maximize $\mu^\top w - \tfrac{\gamma}{2}\, w^\top \Sigma_\lambda w$ over $z$ (Adam on $w = z/\lVert z\rVert_1$).


In [ ]:
z = (torch.randn(N, dtype=torch.float64) / N ** 0.5).clone().requires_grad_(True)
opt = torch.optim.Adam([z], lr=0.05)
for _ in range(10000):
    opt.zero_grad()
    w = l1norm(z)
    loss = -(mu_t @ w - 0.5 * gamma * w @ Sjit_t @ w)
    loss.backward()
    opt.step()

w_util = l1norm(z).detach().numpy()
report("utility", w_util)
s = pd.Series(w_util, index=tickers)
print("\ntop longs:\n", s[s > 1e-4].sort_values(ascending=False).head(6).round(3))
print("\ntop shorts:\n", s[s < -1e-4].sort_values().head(6).round(3))


## Version B — ratio objective

Maximize $\mu^\top w / w^\top \Sigma_\lambda w$ over $z$.


In [ ]:
z = (torch.randn(N, dtype=torch.float64) / N ** 0.5).clone().requires_grad_(True)
opt = torch.optim.Adam([z], lr=0.05)
for _ in range(10000):
    opt.zero_grad()
    w = l1norm(z)
    ratio = (mu_t @ w) / (w @ Sjit_t @ w)
    (-ratio).backward()
    opt.step()

w_rat = l1norm(z).detach().numpy()
report("ratio  ", w_rat)
s = pd.Series(w_rat, index=tickers)
print("\ntop longs:\n", s[s > 1e-4].sort_values(ascending=False).head(6).round(3))
print("\ntop shorts:\n", s[s < -1e-4].sort_values().head(6).round(3))


## Compare


In [ ]:
rows = []
for name, w in [("utility", w_util), ("ratio", w_rat)]:
    rows.append({"objective": name,
                  "gross": np.abs(w).sum(),
                  "net": w.sum(),
                  "longs": int((w > 1e-4).sum()),
                  "shorts": int((w < -1e-4).sum()),
                  "Sharpe": (w @ mu) * 252 / np.sqrt(w @ Sigma @ w * 252)})
pd.DataFrame(rows)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, w) in zip(axes, [("utility", w_util), ("ratio", w_rat)]):
    s = pd.Series(w, index=tickers).sort_values()
    colors = ["tab:red" if v < 0 else "tab:blue" for v in s.values]
    ax.bar(range(len(s)), s.values, color=colors)
    ax.axhline(0, color="k", lw=0.5)
    sh = (w @ mu) * 252 / np.sqrt(w @ Sigma @ w * 252)
    ax.set_title(f"{name}: net={s.sum():+.2f}, Sharpe={sh:.1f}")
    ax.set_ylabel("weight")
plt.tight_layout()


### Takeaway

- **$\lVert w\rVert_1=1$ = gross-exposure constraint.** $w=z/\lVert z\rVert_1$ enforces it for free with signed (long/short) weights; net exposure is whatever the optimizer wants.
- **Utility (γ=2)** → concentrated, net-long book (few positions, net ≈ +0.87). Raise γ to spread out / de-risk; the explicit risk penalty keeps the in-sample Sharpe modest.
- **Ratio** → diversified, near-market-neutral long/short (hundreds of names, net ≈ +0.14) — but its in-sample Sharpe is huge (~100), i.e. overfit. With $\lVert w\rVert_1=1$ the L2 jitter $\lambda\lVert w\rVert^2$ is weak for diversified books ($\lVert w\rVert^2$ is tiny), so the ratio still exploits low-variance directions.
- Both are in-sample; validate out-of-sample. To make the ratio book honest you'd want a **factor-model** $\Sigma$ or a regularizer that bites diversified weights (e.g. penalize positions / gross or net directly).
